##### Copyright 2025 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Zero-Day Vulnerability Finder with Gemma 3

<table align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/google-gemini/gemma-cookbook/blob/main/Gemma/[Gemma_3]Zero_Day_Finder.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
</table>

## Overview

This notebook demonstrates how to build an **agentic security analysis pipeline** using Gemma 3 to assist security researchers in identifying potential vulnerabilities in source code. The system performs structured, multi-pass analysis modeled on real vulnerability research workflows.

**Use cases:**
- Security code review assistance during development
- Educational tool for learning about common vulnerability classes
- First-pass triage of code before manual expert review
- Research into LLM-assisted static analysis

> **Note:** This tool is designed for **defensive security research and education**. All example code snippets in this notebook are intentionally vulnerable and created solely for instructional purposes. Always obtain proper authorization before analyzing third-party code.

### How it works

The pipeline uses a **ReAct (Reasoning + Acting)** pattern to guide Gemma 3 through a structured analysis loop:

1. **Triage** — Identify the language, entry points, and trust boundaries
2. **Pattern scan** — Check for known vulnerability classes (memory safety, injection, logic flaws)
3. **Severity assessment** — Rate exploitability and impact using CVSS-style criteria
4. **Report generation** — Produce a structured finding with remediation guidance

### Vulnerability classes covered

| Class | Examples |
|---|---|
| Memory safety | Buffer overflow, use-after-free, integer overflow |
| Injection | SQL injection, command injection, format string |
| Concurrency | Race conditions, TOCTOU |
| Logic flaws | Auth bypass, insecure deserialization |

## Setup

### Select the Colab runtime

To run Gemma 3 4B, you need a GPU runtime. In the upper-right of the Colab window:
1. Select **▾ (Additional connection options)**
2. Select **Change runtime type**
3. Under **Hardware accelerator**, select **A100 GPU** (recommended) or **L4 GPU**

### Configure Kaggle credentials

To access Gemma model weights, you need a Kaggle account with Gemma access enabled.

1. Open the 🔑 **Secrets** tab in the left panel
2. Create secrets `KAGGLE_USERNAME` and `KAGGLE_KEY`
3. Toggle the buttons to allow notebook access

In [ ]:
import os
from google.colab import userdata

# Note: `userdata.get` is a Colab API. If you're not using Colab, set the env
# vars as appropriate for your system.
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

# Avoid memory fragmentation on JAX backend
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "1.00"

In [ ]:
!pip install -q git+https://github.com/google-deepmind/gemma.git kagglehub

## Load Gemma 3

In [ ]:
import kagglehub

GEMMA_VARIANT = "gemma3-4b-it"
GEMMA_PATH = kagglehub.model_download(f"google/gemma-3/flax/{GEMMA_VARIANT}")

CKPT_PATH = os.path.join(GEMMA_PATH, GEMMA_VARIANT)
TOKENIZER_PATH = os.path.join(GEMMA_PATH, "tokenizer.model")
print("CKPT_PATH:", CKPT_PATH)
print("TOKENIZER_PATH:", TOKENIZER_PATH)

In [ ]:
from gemma import gm

model = gm.nn.Gemma3_4B()
params = gm.ckpts.load_params(CKPT_PATH)
tokenizer = gm.text.Gemma3Tokenizer(TOKENIZER_PATH)

print("Gemma 3 4B loaded successfully.")

## Build the Vulnerability Analysis Agent

The agent follows a **ReAct loop**: it reasons about the code, selects an analysis tool, and incorporates the tool output before proceeding to the next step. This mirrors how a human security researcher works through an unfamiliar codebase.

### Available tools

| Tool | Purpose |
|---|---|
| `scan_memory_safety(code)` | Check for buffer overflows, use-after-free, integer issues |
| `scan_injection(code)` | Check for SQL, command, and format-string injection |
| `scan_concurrency(code)` | Check for race conditions and TOCTOU flaws |
| `scan_auth_logic(code)` | Check for authentication and authorization bypasses |
| `generate_report(findings)` | Compile findings into a structured security report |

In [ ]:
SYSTEM_PROMPT = """\
You are a senior security researcher specializing in vulnerability discovery. \
Your job is to analyze source code for security vulnerabilities that could be \
exploited by an attacker.

You have access to the following analysis tools:

* `scan_memory_safety(code)`: Scans code for memory safety issues such as \
buffer overflows, out-of-bounds reads/writes, use-after-free, double-free, \
and integer overflows.
* `scan_injection(code)`: Scans code for injection vulnerabilities including \
SQL injection, OS command injection, and format-string vulnerabilities.
* `scan_concurrency(code)`: Scans code for concurrency flaws such as race \
conditions, time-of-check-time-of-use (TOCTOU), and improper locking.
* `scan_auth_logic(code)`: Scans code for authentication and authorization \
flaws, insecure defaults, and logic bypasses.
* `generate_report(findings)`: Takes a list of vulnerability findings and \
produces a structured security report with CVSS scores and remediation steps.

Use the following format strictly:

Thought: <your reasoning about which tool to use and why>
Action: <tool_name>(<arguments>)

After receiving a tool result, continue with:

Thought: <reasoning about the result and next step>
Action: <next tool call, or generate_report if analysis is complete>

Always run ALL four scan tools before calling generate_report. \
Be precise, technical, and reference specific line numbers where possible.
"""

START_TURN_USER = "<start_of_turn>user\n"
START_TURN_MODEL = "<start_of_turn>model\n"
END_TURN = "<end_of_turn>\n"

### Implement the analysis tools

In a production system, these tools would call compiled static analyzers (e.g., CodeQL, Semgrep, AddressSanitizer). Here we use Gemma itself as a focused single-pass scanner for each vulnerability class, demonstrating how an LLM can be decomposed into specialized sub-tasks.

In [ ]:
import re
import textwrap


def _gemma_query(prompt: str, max_length: int = 2048) -> str:
    """Run a single-turn query against the loaded Gemma model."""
    sampler = gm.text.ChatSampler(
        model=model,
        params=params,
        tokenizer=tokenizer,
        multi_turn=False,
    )
    return sampler.chat(prompt=prompt)


def scan_memory_safety(code: str) -> str:
    """Scan code for memory safety vulnerabilities."""
    prompt = (
        "You are a memory-safety expert. Analyze the following code for memory "
        "safety vulnerabilities including: buffer overflows, out-of-bounds "
        "reads/writes, use-after-free, double-free, integer overflow leading to "
        "allocation issues, and dangling pointers.\n\n"
        "For each finding, state:\n"
        "- Vulnerability type\n"
        "- Affected lines or function\n"
        "- How an attacker could trigger it\n"
        "- Potential impact\n\n"
        "If no memory safety issues are found, say 'No memory safety issues detected.'\n\n"
        f"Code to analyze:\n```\n{code}\n```"
    )
    return _gemma_query(prompt)


def scan_injection(code: str) -> str:
    """Scan code for injection vulnerabilities."""
    prompt = (
        "You are an injection vulnerability expert. Analyze the following code "
        "for injection vulnerabilities including: SQL injection, OS command "
        "injection, format-string vulnerabilities, path traversal, and "
        "expression language injection.\n\n"
        "For each finding, state:\n"
        "- Vulnerability type\n"
        "- Affected lines or function\n"
        "- The attacker-controlled input path\n"
        "- Potential impact\n\n"
        "If no injection issues are found, say 'No injection vulnerabilities detected.'\n\n"
        f"Code to analyze:\n```\n{code}\n```"
    )
    return _gemma_query(prompt)


def scan_concurrency(code: str) -> str:
    """Scan code for concurrency vulnerabilities."""
    prompt = (
        "You are a concurrency security expert. Analyze the following code for "
        "concurrency vulnerabilities including: race conditions, "
        "time-of-check-time-of-use (TOCTOU) flaws, improper use of locks, "
        "signal handler issues, and double-checked locking bugs.\n\n"
        "For each finding, state:\n"
        "- Vulnerability type\n"
        "- Affected lines or function\n"
        "- The race window\n"
        "- Potential impact\n\n"
        "If no concurrency issues are found, say 'No concurrency vulnerabilities detected.'\n\n"
        f"Code to analyze:\n```\n{code}\n```"
    )
    return _gemma_query(prompt)


def scan_auth_logic(code: str) -> str:
    """Scan code for authentication and authorization logic flaws."""
    prompt = (
        "You are an application security expert specializing in authentication "
        "and authorization. Analyze the following code for: authentication "
        "bypasses, authorization flaws, insecure defaults, privilege escalation "
        "paths, insecure deserialization, and broken access control.\n\n"
        "For each finding, state:\n"
        "- Vulnerability type\n"
        "- Affected lines or function\n"
        "- How an attacker could exploit it\n"
        "- Potential impact\n\n"
        "If no auth/logic issues are found, say 'No authentication or logic vulnerabilities detected.'\n\n"
        f"Code to analyze:\n```\n{code}\n```"
    )
    return _gemma_query(prompt)


def generate_report(findings: str) -> str:
    """Generate a structured security report from accumulated findings."""
    prompt = (
        "You are a security report writer. Given the following vulnerability "
        "findings from multiple analysis passes, produce a structured security "
        "report with:\n\n"
        "1. **Executive Summary** (2-3 sentences, non-technical)\n"
        "2. **Findings Table** with columns: ID | Title | Severity (Critical/High/Medium/Low) | CVSS Score | Status\n"
        "3. **Detailed Findings** — for each vulnerability:\n"
        "   - Description\n"
        "   - Affected Component\n"
        "   - Attack Scenario\n"
        "   - Remediation (specific code fix guidance)\n"
        "4. **Remediation Priority** — ordered list from most to least critical\n\n"
        f"Findings input:\n{findings}"
    )
    return _gemma_query(prompt)


# Map tool names to functions for the agent dispatch loop
TOOLS = {
    "scan_memory_safety": scan_memory_safety,
    "scan_injection": scan_injection,
    "scan_concurrency": scan_concurrency,
    "scan_auth_logic": scan_auth_logic,
    "generate_report": generate_report,
}

print("Analysis tools initialized:", list(TOOLS.keys()))

### Build the ReAct agent loop

The agent loop parses Gemma's `Action: tool_name(args)` outputs, dispatches to the right tool, and feeds the result back as context for the next reasoning step.

In [ ]:
def parse_action(text: str):
    """Extract (tool_name, argument) from a ReAct Action line."""
    match = re.search(r"Action:\s*(\w+)\((.*)\)", text, re.DOTALL)
    if not match:
        return None, None
    tool_name = match.group(1).strip()
    # The argument is everything inside the outer parentheses; strip quotes.
    raw_arg = match.group(2).strip()
    # Remove surrounding quotes if present
    if (raw_arg.startswith('"') and raw_arg.endswith('"')) or (
        raw_arg.startswith("'") and raw_arg.endswith("'")
    ):
        raw_arg = raw_arg[1:-1]
    return tool_name, raw_arg


def run_vulnerability_analysis(source_code: str, max_steps: int = 10) -> str:
    """
    Run the full ReAct vulnerability analysis loop on the given source code.

    Returns the final security report.
    """
    sampler = gm.text.ChatSampler(
        model=model,
        params=params,
        tokenizer=tokenizer,
        multi_turn=True,
    )

    # Initial user message: system prompt + code to analyze
    initial_message = (
        SYSTEM_PROMPT
        + "\n\nAnalyze the following source code for security vulnerabilities:\n\n"
        + f"```\n{source_code}\n```\n\n"
        + "Begin your analysis now."
    )

    conversation = initial_message
    all_findings = []
    final_report = None
    step = 0

    print("=" * 70)
    print("ZERO-DAY VULNERABILITY ANALYSIS")
    print("=" * 70)

    while step < max_steps:
        step += 1
        response = sampler.chat(prompt=conversation if step == 1 else tool_result_msg)

        # Print the model's reasoning
        thought_match = re.search(r"Thought:(.*?)(?=Action:|$)", response, re.DOTALL)
        if thought_match:
            thought = thought_match.group(1).strip()
            print(f"\n[Step {step}] Thought: {thought[:300]}{'...' if len(thought) > 300 else ''}")

        tool_name, tool_arg = parse_action(response)

        if not tool_name or tool_name not in TOOLS:
            # No more tool calls — check if there's a final answer
            print("\n[Agent] Analysis complete.")
            final_report = response
            break

        print(f"[Step {step}] Action: {tool_name}(...)")

        # Dispatch to the appropriate tool
        if tool_name == "generate_report":
            combined_findings = "\n\n".join(all_findings)
            tool_result = TOOLS[tool_name](combined_findings)
            final_report = tool_result
            print("\n[Agent] Report generated.")
            break
        else:
            # The scan tools operate on the original source code
            tool_result = TOOLS[tool_name](source_code)
            all_findings.append(f"### {tool_name} results:\n{tool_result}")
            print(f"[Tool result preview] {tool_result[:200]}...")

        # Feed the tool result back to the model
        tool_result_msg = f"Tool result for {tool_name}:\n{tool_result}"

    return final_report or "\n".join(all_findings)

## Vulnerability Analysis Examples

The following sections demonstrate the analyzer on intentionally vulnerable code snippets, each illustrating a different vulnerability class. These snippets are for **educational purposes only**.

### Example 1: Buffer Overflow + Integer Overflow (C)

This C function contains a classic stack-based buffer overflow triggered by an integer overflow in the size calculation — a pattern seen in many real-world heap allocator vulnerabilities.

In [ ]:
VULNERABLE_C_CODE = """\
#include <string.h>
#include <stdlib.h>
#include <stdio.h>

/* Concatenate two user-supplied buffers into a fixed-size stack buffer. */
int merge_inputs(char *buf1, size_t len1, char *buf2, size_t len2) {
    char result[256];
    size_t total = len1 + len2;          /* (1) Integer overflow if len1+len2 > SIZE_MAX */

    if (total > 256) {                   /* (2) Overflow bypasses this check */
        return -1;
    }

    memcpy(result, buf1, len1);          /* (3) Stack buffer overflow */
    memcpy(result + len1, buf2, len2);
    result[total] = '\\0';

    printf(result);                      /* (4) Format-string vulnerability */
    return 0;
}

/* Frees a node but keeps a pointer to it — caller may still access freed memory. */
typedef struct Node { int value; struct Node *next; } Node;

void delete_node(Node **head, Node *target) {
    Node *cur = *head;
    while (cur && cur->next != target) cur = cur->next;
    if (cur) cur->next = target->next;
    free(target);                        /* (5) Caller's stale pointer: use-after-free risk */
}
"""

report_c = run_vulnerability_analysis(VULNERABLE_C_CODE)
print("\n" + "=" * 70)
print("FINAL REPORT")
print("=" * 70)
print(report_c)

### Example 2: SQL Injection + Command Injection (Python)

This Python web handler passes unsanitized user input directly into a SQL query and a shell command — two of the most commonly exploited vulnerability classes in web applications.

In [ ]:
VULNERABLE_PYTHON_CODE = """\
import sqlite3
import subprocess
from flask import Flask, request, jsonify

app = Flask(__name__)
DB_PATH = "/var/data/app.db"


@app.route("/user")
def get_user():
    # (1) SQL injection: username comes directly from query string
    username = request.args.get("username", "")
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    query = f"SELECT * FROM users WHERE username = '{username}'"
    cursor.execute(query)
    rows = cursor.fetchall()
    conn.close()
    return jsonify(rows)


@app.route("/ping")
def ping_host():
    # (2) OS command injection: host is attacker-controlled
    host = request.args.get("host", "127.0.0.1")
    result = subprocess.check_output(f"ping -c 1 {host}", shell=True)
    return result.decode()


@app.route("/admin")
def admin_panel():
    # (3) Authentication bypass: relies on attacker-controlled header
    role = request.headers.get("X-User-Role", "guest")
    if role == "admin":
        return "Welcome, admin!"
    return "Access denied", 403
"""

report_python = run_vulnerability_analysis(VULNERABLE_PYTHON_CODE)
print("\n" + "=" * 70)
print("FINAL REPORT")
print("=" * 70)
print(report_python)

### Example 3: Race Condition + TOCTOU (Python)

This file-processing service checks whether a file is safe before processing it, but does so in a non-atomic way — creating a classic TOCTOU (time-of-check time-of-use) race window.

In [ ]:
VULNERABLE_RACE_CODE = """\
import os
import threading

UPLOAD_DIR = "/var/uploads"
SAFE_EXTENSIONS = {".txt", ".csv", ".json"}

# Shared counter with no locking
processed_count = 0


def is_safe_file(path: str) -> bool:
    """Return True if the file has an allowed extension and is not a symlink."""
    _, ext = os.path.splitext(path)
    return ext.lower() in SAFE_EXTENSIONS and not os.path.islink(path)
    # (1) TOCTOU: attacker can swap symlink between is_safe_file() and open()


def process_file(filename: str):
    global processed_count

    filepath = os.path.join(UPLOAD_DIR, filename)

    # CHECK
    if not is_safe_file(filepath):
        print(f"Rejected: {filename}")
        return

    # USE  <-- race window between check and use
    with open(filepath, "r") as f:  # (2) Opens attacker-substituted symlink
        data = f.read()

    # (3) Race condition: unprotected increment across threads
    processed_count += 1
    print(f"Processed {filename} (total: {processed_count})")


def handle_uploads(filenames):
    threads = [threading.Thread(target=process_file, args=(fn,)) for fn in filenames]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
"""

report_race = run_vulnerability_analysis(VULNERABLE_RACE_CODE)
print("\n" + "=" * 70)
print("FINAL REPORT")
print("=" * 70)
print(report_race)

## Single-Pass Targeted Scan

For quick analysis, you can invoke individual scan tools directly without running the full agent loop. This is useful when you already know the likely vulnerability class.

In [ ]:
from IPython.display import display, Markdown

# Targeted memory safety scan on the C snippet
result = scan_memory_safety(VULNERABLE_C_CODE)
display(Markdown("### Memory Safety Scan Results\n\n" + result))

In [ ]:
# Targeted injection scan on the Python web handler
result = scan_injection(VULNERABLE_PYTHON_CODE)
display(Markdown("### Injection Vulnerability Scan Results\n\n" + result))

## Analyze Your Own Code

Paste your own code snippet below and run a full analysis. This is intended for use on code **you own or have authorization to test**.

In [ ]:
# @title Enter code to analyze { display-mode: "form" }

YOUR_CODE = """  # @param {type: "string"}
// Paste your code here
"""

if YOUR_CODE.strip() and YOUR_CODE.strip() != "// Paste your code here":
    custom_report = run_vulnerability_analysis(YOUR_CODE)
    display(Markdown("## Security Analysis Report\n\n" + custom_report))
else:
    print("Add your code to the YOUR_CODE variable above and re-run this cell.")

## Summary

This notebook demonstrated how to build a **ReAct-style agentic vulnerability analysis pipeline** with Gemma 3. Key takeaways:

- **Decomposition works well**: Breaking the analysis into specialized single-class scans (memory, injection, concurrency, logic) produces more precise findings than a single broad prompt.
- **ReAct grounding**: Having the model explicitly state its reasoning before each tool call reduces hallucinations and makes the analysis process auditable.
- **LLM as a first-pass triage layer**: Gemma can surface candidate vulnerabilities quickly, allowing human experts to focus their time on the most suspicious areas.
- **Limitations**: LLM-based analysis can produce false positives and may miss subtle vulnerabilities that require deep dataflow analysis. Always validate findings with dedicated static analysis tools (CodeQL, Semgrep, AddressSanitizer) and manual review.

### Next steps

- Fine-tune Gemma on a labeled vulnerability dataset (e.g., Big-Vul, CVEfixes) to improve precision
- Integrate real static analysis tools as backends for the scan functions
- Extend the agent with a `search_cve_database(cwe_id)` tool to link findings to known CVEs
- Build a CI/CD integration that runs the pipeline on every pull request